# 02 · Train tokenizer and tokenize data (Kaggle CPU session)

Needs the text shards uploaded by `01_prepare_data`. Trains the shared 32k BPE tokenizer, reports fertility, encodes all text into uint16 token files and uploads everything to the HF dataset repo.

**Setup:** Settings → Accelerator: *None*, Internet: *On*. Add-ons → Secrets: `HF_TOKEN`.

In [ ]:
HF_USER = "DeerKaraca"
HUB_REPO = f"{HF_USER}/bilingual-slm-data"

In [ ]:
!git clone -q https://github.com/DeerKaraca/bilingual-slm.git
%cd bilingual-slm
!pip install -q -e ".[data]"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, snapshot_download

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
snapshot_download(HUB_REPO, repo_type="dataset", allow_patterns="text/*", local_dir="data")

In [ ]:
!python -m slm.tokenizer --text-dir data/text --mb-per-lang 300 --out assets/tokenizer.json
!python -m slm.eval.fertility --text-dir data/text --tokenizer assets/tokenizer.json --out reports/tokenizer.md

In [ ]:
!python -m slm.data.tokenize --text-dir data/text --tokenizer assets/tokenizer.json --out data/tokens

In [ ]:
api = HfApi()
api.upload_folder(repo_id=HUB_REPO, repo_type="dataset", folder_path="data/tokens", path_in_repo="tokens")
api.upload_file(repo_id=HUB_REPO, repo_type="dataset", path_or_fileobj="assets/tokenizer.json", path_in_repo="tokenizer.json")
api.upload_file(repo_id=HUB_REPO, repo_type="dataset", path_or_fileobj="reports/tokenizer.md", path_in_repo="reports/tokenizer.md")